# Figure 4. Event-scale evolution of a representative precursor cluster

This notebook draws a six-panel event-scale figure:

- **a** waveform + labelled debris-flow interval
- **b** 1-min RMS envelope
- **c** PSD evolution
- **d** DEC state sequence, with PCS cluster highlighted
- **e** PCS cluster occurrence and cumulative count
- **f** representative spectrograms of the PCS precursor cluster

Change `EVENT` in the first code cell to redraw another retained debris-flow event.

## 1. Imports, paths and parameters

In [ ]:
from pathlib import Path
import csv
import re
import zipfile
import xml.etree.ElementTree as ET

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from obspy import UTCDateTime, read
from scipy.signal import welch

PROJECT_ROOT = Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben")
EVENT = "event3"

DATA_DIR = PROJECT_ROOT / "data"
DEC_RESULT_DIR = PROJECT_ROOT / r"uilt_oneday_all\02clustering\result\DEC_result"
PARAM_DIR = DEC_RESULT_DIR / "all_parement"
METRICS_FILE = DEC_RESULT_DIR / r"precursor_curve_metrics\all_cluster_metrics_by_sheet.xlsx"
OUT_DIR = PROJECT_ROOT / r"uilt_oneday_all\uilt_paper_figure\Fig4_event_scale"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRE_H = 24
POST_H = 2
HP = 1.0
RMS_WIN = 60
PSD_WIN = 600
PSD_FREQ = (1, 50)
N_SPEC = 5
EPS = 1e-20

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7.5,
    "xtick.labelsize": 6.8,
    "ytick.labelsize": 6.8,
    "legend.fontsize": 6.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.65,
    "xtick.major.width": 0.65,
    "ytick.major.width": 0.65,
    "legend.frameon": False,
})

## 2. Lightweight Excel readers

In [ ]:
def event_order(name):
    match = re.search(r"event(\d+)", str(name))
    return int(match.group(1)) if match else 10**9


def col_to_index(cell_ref):
    letters = re.match(r"([A-Z]+)", cell_ref).group(1)
    value = 0
    for char in letters:
        value = value * 26 + ord(char) - ord("A") + 1
    return value - 1


def text_from_si(si_node, ns):
    return "".join(node.text or "" for node in si_node.findall(".//m:t", ns))


def dedupe_header(header):
    seen = {}
    out = []
    for name in header:
        key = str(name).strip()
        seen[key] = seen.get(key, 0) + 1
        out.append(key if seen[key] == 1 else f"{key}.{seen[key]}")
    return out


def read_xlsx_sheet(path, sheet_name=None):
    ns = {
        "m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
        "rel": "http://schemas.openxmlformats.org/package/2006/relationships",
    }
    with zipfile.ZipFile(path) as zf:
        shared = []
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            shared = [text_from_si(si, ns) for si in root.findall("m:si", ns)]

        if sheet_name is None:
            sheet_xml = "xl/worksheets/sheet1.xml"
        else:
            workbook = ET.fromstring(zf.read("xl/workbook.xml"))
            rels = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
            rel_map = {rel.attrib["Id"]: rel.attrib["Target"] for rel in rels.findall("rel:Relationship", ns)}
            target = None
            for sheet in workbook.findall(".//m:sheet", ns):
                if sheet.attrib["name"] == sheet_name:
                    target = rel_map[sheet.attrib[f"{{{ns['r']}}}id"]]
                    break
            if target is None:
                raise ValueError(f"Sheet not found: {sheet_name}")
            target = target.lstrip("/")
            sheet_xml = target if target.startswith("xl/") else f"xl/{target}"

        root = ET.fromstring(zf.read(sheet_xml))

    rows = []
    for row in root.findall(".//m:sheetData/m:row", ns):
        values = []
        for cell in row.findall("m:c", ns):
            idx = col_to_index(cell.attrib["r"])
            while len(values) <= idx:
                values.append("")
            cell_type = cell.attrib.get("t")
            if cell_type == "inlineStr":
                values[idx] = "".join(node.text or "" for node in cell.findall(".//m:t", ns))
                continue
            value_node = cell.find("m:v", ns)
            if value_node is None:
                values[idx] = ""
                continue
            raw = value_node.text
            if cell_type == "s":
                values[idx] = shared[int(raw)]
            else:
                try:
                    values[idx] = float(raw)
                except Exception:
                    values[idx] = raw
        if any(str(v).strip() for v in values):
            rows.append(values)

    header = dedupe_header(rows[0])
    records = []
    for values in rows[1:]:
        values = values + [""] * (len(header) - len(values))
        records.append(dict(zip(header, values[:len(header)])))
    return records


def find_event_table():
    for path in DATA_DIR.glob("*.xlsx"):
        if path.name.startswith("~$"):
            continue
        rows = read_xlsx_sheet(path)
        if rows and {"Manual_Label_start", "Manual_Label_end", "SAC_File_Path"}.issubset(rows[0]):
            return path, rows
    raise FileNotFoundError("Could not find event table with Manual_Label_start/Manual_Label_end/SAC_File_Path.")


def as_float(value):
    try:
        return float(value)
    except Exception:
        return np.nan


def parse_time(value):
    text = str(value).strip()
    if re.fullmatch(r"\d+(\.\d+)?", text):
        return UTCDateTime((float(text) - 25569.0) * 86400.0)
    return UTCDateTime(text)

## 3. PCS-selected precursor cluster

In [ ]:
def zscore(values):
    values = np.asarray(values, dtype=float)
    mean = np.nanmean(values)
    std = np.nanstd(values)
    if not np.isfinite(std) or std <= 0:
        return np.zeros_like(values)
    return (values - mean) / std


def add_pcs_scores(records):
    scored = []
    for event_name in sorted({str(row["event"]) for row in records}, key=event_order):
        rows = [row for row in records if str(row["event"]) == event_name]
        zz = zscore([as_float(r["z_final"]) for r in rows])
        za = zscore([as_float(r["rate_acceleration"]) for r in rows])
        zc = zscore([as_float(r["c_value"]) for r in rows])
        zg = zscore([as_float(r["terminal_growth_ratio"]) for r in rows])
        for i, row in enumerate(rows):
            row = row.copy()
            row["pcs"] = float(zz[i] + za[i] - zc[i] + zg[i])
            scored.append(row)
    return scored


def select_pcs_precursors(scored, z_thr=2.0, c_thr=1.0):
    selected = {}
    for event_name in sorted({str(row["event"]) for row in scored}, key=event_order):
        rows = [row for row in scored if str(row["event"]) == event_name]
        candidates = [
            row for row in rows
            if as_float(row["z_final"]) > z_thr
            and as_float(row["c_value"]) < c_thr
            and int(as_float(row["terminal_growth_flag"])) == 1
        ]
        if not candidates:
            selected[event_name] = None
            continue
        selected[event_name] = sorted(
            candidates,
            key=lambda row: (
                as_float(row["pcs"]),
                as_float(row["z_final"]),
                as_float(row["terminal_growth_ratio"]),
                as_float(row["sample_count"]),
            ),
            reverse=True,
        )[0]
    return selected


metrics = read_xlsx_sheet(METRICS_FILE, "all_metrics_wide")
selected = select_pcs_precursors(add_pcs_scores(metrics))
selected_row = selected.get(EVENT)
if selected_row is None:
    raise ValueError(f"{EVENT} has no PCS-selected precursor cluster. Choose another event.")

PCS_CLUSTER = int(as_float(selected_row["cluster"]))
print(EVENT, "PCS cluster:", PCS_CLUSTER, "PCS:", round(as_float(selected_row["pcs"]), 3), "samples:", int(as_float(selected_row["sample_count"])))

## 4. Event waveform, RMS and PSD

In [ ]:
event_table_path, event_rows = find_event_table()
event_row = next(
    row for row in event_rows
    if str(row.get("Event1", "")).strip() == EVENT
    or str(row.get("Event", "")).strip() == EVENT
    or str(row.get("Event.2", "")).strip() == EVENT
)

start = parse_time(event_row["Manual_Label_start"])
end = parse_time(event_row["Manual_Label_end"])
sac_path = Path(str(event_row["SAC_File_Path"]))
print("event table:", event_table_path)
print("SAC:", sac_path, "exists:", sac_path.exists())

tr = read(str(sac_path), starttime=start - PRE_H * 3600, endtime=start + POST_H * 3600)[0]
tr.trim(start - PRE_H * 3600, start + POST_H * 3600, pad=True, fill_value=0)
tr.detrend("demean")
tr.detrend("linear")
tr.taper(max_percentage=0.01, type="hann")
tr.filter("highpass", freq=HP, corners=4, zerophase=True)
fs = float(tr.stats.sampling_rate)
data = tr.data.astype(float)


def rms_curve(data, fs):
    n = int(RMS_WIN * fs)
    m = len(data) // n
    y = data[: m * n].reshape(m, n)
    rms = np.sqrt(np.mean(y ** 2, axis=1))
    x = -PRE_H + (np.arange(m) + 0.5) * RMS_WIN / 3600
    return x, rms


def psd_evolution(data, fs):
    n = int(PSD_WIN * fs)
    step = n // 2
    starts = np.arange(0, len(data) - n + 1, step)
    cols = []
    xs = []
    freqs = None
    for s in starts:
        f, p = welch(data[s:s + n], fs=fs, nperseg=min(4096, n), noverlap=min(2048, n // 2))
        freqs = f
        cols.append(10 * np.log10(p + EPS))
        xs.append(-PRE_H + (s + n / 2) / fs / 3600)
    z = np.column_stack(cols)
    keep = (freqs >= PSD_FREQ[0]) & (freqs <= PSD_FREQ[1])
    return np.asarray(xs), freqs[keep], z[keep]


x_rms, rms = rms_curve(data, fs)
rms_norm = rms / np.nanpercentile(rms, 99)

pre_n = int(PRE_H * 3600 * fs)
x_psd, f_psd, z_psd = psd_evolution(data[:pre_n], fs)

stride = max(1, len(data) // 60000)
x_wave = -PRE_H + np.arange(0, len(data), stride) / fs / 3600
y_wave = data[::stride]
y_wave = y_wave / np.nanpercentile(np.abs(y_wave), 99)

event_end_h = min(max((end - start) / 3600, 0), POST_H)
print("sampling rate:", fs, "Hz")

## 5. DEC labels, precursor occurrence and representative spectrograms

In [ ]:
npz = np.load(PARAM_DIR / f"{EVENT}_all_parement.npz", allow_pickle=True)
labels = np.asarray(npz["y_pred_last"]).astype(int)
spectral = np.asarray(npz["spectral"])
distances = np.asarray(npz["distances"])
n_clusters = int(np.asarray(npz["n_clusters"]))

relative_hours = np.arange(len(labels), dtype=float) / 60.0 - PRE_H
precursor_mask = labels == PCS_CLUSTER
minute_counts = precursor_mask.astype(int)
cumulative = np.cumsum(minute_counts)

cluster_ids = np.arange(n_clusters)
colors = plt.get_cmap("tab10")(cluster_ids % 10)
precursor_color = "#D95F02"


def self_norm(arr):
    arr = np.squeeze(arr).astype(float)
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-12)


def representative_spectrogram_indices(labels, distances, cluster_id, n=5):
    idx = np.where(labels == cluster_id)[0]
    if len(idx) == 0:
        return np.array([], dtype=int)
    if distances.ndim == 2 and cluster_id < distances.shape[1]:
        idx = idx[np.argsort(distances[idx, cluster_id])]
    return idx[: min(n, len(idx))]


spec_idx = representative_spectrogram_indices(labels, distances, PCS_CLUSTER, N_SPEC)
print("n_clusters:", n_clusters, "PCS samples:", int(precursor_mask.sum()), "spectrogram examples:", spec_idx.tolist())

## 6. Draw and save Figure 4

In [ ]:
fig = plt.figure(figsize=(7.2, 9.35))
gs = fig.add_gridspec(6, 1, height_ratios=[0.75, 0.75, 1.45, 0.70, 0.90, 1.28], hspace=0.66)
ax_a = fig.add_subplot(gs[0])
ax_b = fig.add_subplot(gs[1], sharex=ax_a)
ax_c = fig.add_subplot(gs[2])
ax_d = fig.add_subplot(gs[3])
ax_e = fig.add_subplot(gs[4], sharex=ax_d)
spec_grid = gs[5].subgridspec(1, max(1, len(spec_idx)), wspace=0.08)
ax_f = [fig.add_subplot(spec_grid[0, i]) for i in range(max(1, len(spec_idx)))]

# a. waveform
ax_a.plot(x_wave, y_wave, color="0.22", lw=0.35)
ax_a.axvspan(0, event_end_h, color="#C23B2E", alpha=0.16, lw=0)
ax_a.axvline(0, color="#B83A2E", lw=0.9)
ax_a.set_xlim(-PRE_H, POST_H)
ax_a.set_ylabel("Waveform\n(norm.)")
ax_a.set_title(f"{EVENT}: waveform and labelled debris-flow interval", loc="left")

# b. RMS
ax_b.plot(x_rms, rms_norm, color="#2B5C8A", lw=0.75)
ax_b.fill_between(x_rms, 0, rms_norm, color="#2B5C8A", alpha=0.18, lw=0)
ax_b.axvspan(0, event_end_h, color="#C23B2E", alpha=0.16, lw=0)
ax_b.axvline(0, color="#B83A2E", lw=0.9)
ax_b.set_ylabel("RMS\n(norm.)")
ax_b.set_title("1-min RMS envelope", loc="left")

# c. PSD
im_psd = ax_c.pcolormesh(
    x_psd, f_psd, z_psd,
    shading="auto",
    cmap="magma",
    vmin=np.nanpercentile(z_psd, 5),
    vmax=np.nanpercentile(z_psd, 98),
)
ax_c.axvline(0, color="#B83A2E", lw=0.9)
ax_c.set_xlim(-PRE_H, 0)
ax_c.set_ylim(*PSD_FREQ)
ax_c.set_ylabel("Frequency (Hz)")
ax_c.set_title("Pre-event PSD evolution", loc="left")
cb = fig.colorbar(im_psd, ax=ax_c, pad=0.012, fraction=0.030)
cb.set_label("PSD (dB)")

# d. DEC state sequence
for cluster_id in cluster_ids:
    mask = labels == cluster_id
    color = precursor_color if cluster_id == PCS_CLUSTER else colors[cluster_id]
    alpha = 0.95 if cluster_id == PCS_CLUSTER else 0.28
    size = 8 if cluster_id == PCS_CLUSTER else 3
    ax_d.scatter(relative_hours[mask], labels[mask], s=size, color=color, alpha=alpha, linewidth=0, rasterized=True)
ax_d.axvline(0, color="#B83A2E", lw=0.9)
ax_d.set_xlim(-PRE_H, 0)
ax_d.set_yticks(cluster_ids)
ax_d.set_ylabel("DEC\ncluster")
ax_d.set_title(f"DEC state sequence; PCS cluster C{PCS_CLUSTER} highlighted", loc="left")
ax_d.grid(axis="y", alpha=0.15)

# e. precursor occurrence and cumulative count
ax_e.vlines(relative_hours[precursor_mask], 0, 1, color=precursor_color, alpha=0.40, lw=0.45, label="Occurrence")
ax_e.set_ylim(0, 1.05)
ax_e.set_yticks([0, 1])
ax_e.set_ylabel("PCS C\nocc.")
ax_e.axvline(0, color="#B83A2E", lw=0.9)
ax_e2 = ax_e.twinx()
ax_e2.plot(relative_hours, cumulative, color="black", lw=1.2, label="Cumulative")
ax_e2.set_ylabel("Cumulative\ncount")
ax_e.set_title("PCS-cluster occurrence and cumulative count", loc="left")
ax_e.set_xlabel("Time before debris-flow initiation (h)")

# f. representative spectrograms
if len(spec_idx) == 0:
    ax_f[0].axis("off")
    ax_f[0].text(0.5, 0.5, "No PCS-cluster samples", ha="center", va="center")
    im_spec = None
else:
    for i, idx in enumerate(spec_idx):
        arr = self_norm(spectral[idx])
        im_spec = ax_f[i].imshow(arr, aspect="auto", origin="lower", cmap="Reds", vmin=0, vmax=1, extent=[0, 60, 0, 50])
        ax_f[i].set_title(f"C{PCS_CLUSTER}, t={relative_hours[idx]:.1f} h", pad=2)
        ax_f[i].set_xlabel("Time (s)")
        if i == 0:
            ax_f[i].set_ylabel("Frequency (Hz)")
        else:
            ax_f[i].set_yticklabels([])
        ax_f[i].set_xticks([0, 30, 60])
        ax_f[i].set_yticks([0, 25, 50])
        for spine in ax_f[i].spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.45)
            spine.set_color("0.35")
    cb2 = fig.colorbar(im_spec, ax=ax_f, orientation="horizontal", fraction=0.08, pad=0.20)
    cb2.set_label("Self-normalized spectral amplitude")

for label, ax in zip(["a", "b", "c", "d", "e"], [ax_a, ax_b, ax_c, ax_d, ax_e]):
    ax.text(-0.075, 1.07, label, transform=ax.transAxes, fontsize=11, fontweight="bold", va="top", ha="left")
ax_f[0].text(-0.18, 1.28, "f", transform=ax_f[0].transAxes, fontsize=11, fontweight="bold", va="top", ha="left")
ax_f[0].text(0, 1.28, "Representative spectrograms of the PCS precursor cluster", transform=ax_f[0].transAxes, fontsize=8, va="top", ha="left")

for ax in [ax_a, ax_b]:
    ax.tick_params(labelbottom=False)
for ax in [ax_a, ax_b, ax_c, ax_d, ax_e]:
    ax.grid(alpha=0.14)

out = OUT_DIR / f"figure4_event_scale_{EVENT}_pcsC{PCS_CLUSTER}.png"
fig.savefig(out, dpi=500, bbox_inches="tight")
plt.close(fig)
print(out)

## 7. Preview

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(out)))